# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
alchemy_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy1k_0405'
aqsol_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol_0405'
orderly_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction'
orderly_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-retrosynthesis'
presto_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction_0405'
presto_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-retrosynthesis_0405'

In [13]:
# load data
alchemy_data = datasets.load_from_disk(alchemy_path)
aqsol_data = datasets.load_from_disk(aqsol_path)
orderly_forward_data = datasets.load_from_disk(orderly_forward_path)
presto_forward_data = datasets.load_from_disk(presto_forward_path)
orderly_retro_data = datasets.load_from_disk(orderly_retro_path)
presto_retro_data = datasets.load_from_disk(presto_retro_path)


In [14]:
ood_data = datasets.concatenate_datasets(
    [
        alchemy_data,
        aqsol_data,
        orderly_forward_data,
        orderly_retro_data,
        presto_forward_data,
        presto_retro_data,
    ]
)

In [15]:
ood_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 7929
})

In [16]:
ood_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood_0411'
)

Saving the dataset (1/1 shards): 100%|██████████| 7929/7929 [00:01<00:00, 4399.28 examples/s] 


In [ ]:
test_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_lumo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_lumo_gap_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_retrosynthesis_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_reagent_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-text2mol_0219',
]

train_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_lumo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_lumo_gap_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_retrosynthesis_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_reagent_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-text2mol_0219',
]


In [22]:
list_ablation_test_data = []
for p in test_paths:
    list_ablation_test_data.append(datasets.load_from_disk(p))
ood_data = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood_0411')
list_ablation_test_data.append(ood_data)

concat_ablation_test_data = datasets.concatenate_datasets(list_ablation_test_data)
concat_ablation_test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 27508
})

In [25]:
concat_ablation_test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ablation_0411'
)

Saving the dataset (1/1 shards): 100%|██████████| 27508/27508 [00:07<00:00, 3893.32 examples/s]


In [24]:
list_ablation_train_data = []
for p in train_paths:
    list_ablation_train_data.append(datasets.load_from_disk(p))
concat_ablation_train_data = datasets.concatenate_datasets(list_ablation_train_data)
concat_ablation_train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 854234
})

In [ ]:
concat_ablation_train_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_ablation_0411'
)

Saving the dataset (1/14 shards):  13%|█▎        | 109017/854234 [00:15<00:37, 19818.30 examples/s]

In [17]:
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ablation_0224'
)
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_validation_ablation_0224'
)

Saving the dataset (0/1 shards):   0%|          | 0/20205 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 20205/20205 [00:01<00:00, 11982.98 examples/s]


In [14]:
train_data = datasets.concatenate_datasets(
    [datasets.load_from_disk(path) for path in train_paths]
)

In [15]:
train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 361115
})

In [18]:
train_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_ablation_0224'
)

Saving the dataset (0/7 shards):   0%|          | 0/361115 [00:00<?, ? examples/s]

Saving the dataset (7/7 shards): 100%|██████████| 361115/361115 [00:28<00:00, 12500.32 examples/s]
